In [4]:
!pip install -q --upgrade pip
!pip install -q \
streamlit \
chromadb \
google-generativeai \
python-dotenv \
pyngrok

In [6]:
import os
import google.generativeai as genai

from kaggle_secrets import UserSecretsClient

secret=UserSecretsClient()

API_KEY=secret.get_secret(
"GEMINI_API_KEY"
)

genai.configure(
api_key=API_KEY
)

print(
"Gemini Connected"
)

Gemini Connected


In [7]:
import os

folder="/kaggle/working/company_docs"

os.makedirs(
folder,
exist_ok=True
)

docs={

"vacation.txt":
"""
Employees receive
20 vacation days.
""",

"remote.txt":
"""
Employees may work
from home 3 days.
""",

"leave.txt":
"""
Employees receive
90 maternity leave days.
"""
}

for name,text in docs.items():

    with open(
f"{folder}/{name}",
"w"
) as f:

        f.write(text)

print(
"Docs Ready"
)

Docs Ready


In [8]:
import chromadb

client=chromadb.PersistentClient(

path=
"/kaggle/working/chroma_db"

)

collection=client.get_or_create_collection(

name=
"company_docs"

)

print(
"Database Ready"
)

Database Ready


In [11]:
def get_embedding(text):

    response=genai.embed_content(

        model=
"models/gemini-embedding-001",

        content=text

    )

    return response[
"embedding"
]

In [12]:
for i,file in enumerate(

os.listdir(folder)

):

    with open(

f"{folder}/{file}"

    ) as f:

        text=f.read()

    emb=get_embedding(
text
)

    collection.add(

ids=[str(i)],

documents=[text],

embeddings=[emb]

    )

print(
"Indexed"
)

Indexed


In [13]:
model=genai.GenerativeModel(
"gemini-2.5-flash"
)

def rag(query):

    emb=get_embedding(
query
)

    results=collection.query(

query_embeddings=[emb],

n_results=2

)

    context="\n".join(

results[
"documents"
][0]

)

    prompt=f"""

Answer ONLY
using context.

Context:
{context}

Question:
{query}

"""

    return model.generate_content(
prompt
).text

In [14]:
%%writefile app.py

import streamlit as st
import chromadb
import google.generativeai as genai

st.set_page_config(
page_title="AI Assistant"
)

client=chromadb.PersistentClient(
"/kaggle/working/chroma_db"
)

collection=client.get_collection(
"company_docs"
)

model=genai.GenerativeModel(
"gemini-2.5-flash"
)


def ask(q):

    result=collection.query(
query_texts=[q]
)

    context="\n".join(
result["documents"][0]
)

    prompt=f"""

Context:
{context}

Question:
{q}

"""

    return model.generate_content(
prompt
).text


st.title(
"Company Assistant"
)

if "messages" not in st.session_state:

    st.session_state.messages=[]


for m in st.session_state.messages:

    with st.chat_message(
m["role"]
):

        st.write(
m["content"]
)


if q:=st.chat_input():

    st.session_state.messages.append(

{
"role":"user",
"content":q
}

)

    answer=ask(q)

    st.session_state.messages.append(

{
"role":"assistant",
"content":answer
}

)

    st.rerun()

Writing app.py


In [ ]:
!streamlit run app.py \
--server.port 8501 \
--server.address 0.0.0.0



2026-05-24 16:18:27.965 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://34.169.154.65:8501

